# laser_driver_v7 — Differential-Thermometer Uncertainty Budget — **fiber Bragg grating** variant

Same instrument, same ratiometric framing, **microring resonators replaced by commercially available fiber Bragg gratings (FBGs)**.

Two gratings are written into **one length of PM fiber**, a few centimetres apart, with a thermal barrier between them. The beat **Δν = ν_B − ν_A = S·(T_B − T_A)** still measures the *local* temperature difference, and the quantity of interest is still a **material-defined ratio** acquired over a **τ ≈ 60 s** window — so the scale factor S and slow common-mode terms still **cancel**, and only **within-window stability** matters.

**Two commercial device classes are budgeted side by side**, because both are catalogue items and they are ~50× apart:

| Class | Linewidth | Lock scheme | Typical source |
|---|---|---|---|
| **π-FBG** — π-phase-shifted grating, narrow transmission notch | **15–50 MHz** (FWHM) | true dispersive **PDH** | TeraXion, AOS GmbH, O/E Land, Technica |
| **Standard telecom FBG** — uniform apodised reflector | **0.1–0.3 nm ≈ 12–37 GHz** | side-of-fringe | any FBG house (~$100) |

What this notebook does, in the same order as the microring version:
1. defines acronyms and the FBG-specific symbols,
2. states what changed physically (and what that costs / buys),
3. keeps the full contributor budget in **hertz** with **kelvin** twins (÷ S),
4. folds in **ratiometric cancellation**,
5. adds the two FBG-specific inputs that decide the floor — **fiber thermodynamic noise** and **parasitic-etalon fringe** — in place of the microring notebook's single RAM input,
6. computes the **Allan deviation out to 600 s** with the 60 s point marked,
7. carries the **square-law voltage twin** through with the FBG's own S and full scale.

All uncertainties are 1σ on Δν; **S = dν/dT ≈ −1.25 GHz/K** for silica FBG at 1550 nm (10 pm/K), versus 2 GHz/K for the SiN ring.

## 1. Acronyms, symbols, and definitions

Everything from the microring notebook still applies. Only the resonator table changes; the FBG-specific additions are marked **new**.

### Optical / grating
| Term | Definition |
|---|---|
| CW | Continuous-wave — constant (non-pulsed) optical output |
| DFB | Distributed-feedback — single-mode semiconductor laser |
| PD | Photodiode |
| InGaAs | Indium gallium arsenide — 1550 nm photodiode material |
| MPD | Monitor photodiode — laser back-facet power monitor |
| TEC | Thermoelectric cooler — Peltier element for temperature control |
| NTC | Negative temperature coefficient — thermistor |
| **FBG** | **new** — fiber Bragg grating; periodic index modulation in the fiber core |
| **π-FBG** | **new** — π-phase-shifted FBG; a defect state opens a narrow transmission notch inside the stop band |
| **λ_B** | **new** — Bragg wavelength, λ_B = 2·n_eff·Λ |
| **Λ** | **new** — grating pitch |
| **n_eff** | **new** — effective modal index (≈1.4468 in SMF-28 at 1550 nm) |
| **L_g** | **new** — grating length (sets the minimum achievable linewidth) |
| **PM / PANDA** | **new** — polarisation-maintaining fiber |
| **APC** | **new (optical)** — angled physical contact connector (8°), kills back-reflection |
| Q | Quality factor — Q = ν₀ / FWHM |
| FSR | Free spectral range — mode spacing (of a *parasitic* fiber etalon here, not of the grating) |

### Modulation and locking
| Term | Definition |
|---|---|
| PDH | Pound–Drever–Hall locking |
| SOF | **new** — side-of-fringe (intensity) locking, the fallback for a broad grating |
| RF | Radio frequency |
| FM / AM | Frequency / amplitude modulation |
| RAM | Residual amplitude modulation — unwanted AM accompanying FM/PM |
| EOM | Electro-optic modulator |
| DDS | Direct digital synthesis (AD9833) |
| LO | Local oscillator |
| ACC / APC | Automatic current control / automatic power control |
| DAC / ADC | Digital-to-analog / analog-to-digital converter |
| β | Modulation index |
| Ω | Modulation frequency |

### Detection, readout, metrology
Unchanged from the microring notebook (TIA, AAF, SNR, SFDR, BW, PSD, RIN, PLL, FWHM, FFT, TCXO/OCXO, GPSDO, CMRR, GUM, Type A/B, u_c, U, k, R&R, repeatability, reproducibility, Allan deviation, ratiometric).

### Symbols
| Symbol | Meaning | Ring value | **FBG value** |
|---|---|---|---|
| ν₀ | Optical carrier frequency at 1550 nm | 193.4 THz | 193.4 THz |
| Δν | Beat (difference) frequency, ν_B − ν_A | — | — |
| S = dν/dT | Frequency–temperature sensitivity | 2 GHz/K | **1.25 GHz/K** (10 pm/K) |
| **S_ε = dν/dε** | **new** — frequency–strain sensitivity | (die-mounted, small) | **−150 MHz/µε** (1.2 pm/µε) |
| FWHM | Reference linewidth | 1.9 GHz (Q = 1e5) | **50 MHz (π-FBG) / 25 GHz (standard)** |
| ε | Fractional mismatch of the two references' temperature coefficients | 1e-4 (same die) | **5e-4 (same fiber, same phase mask)** |
| **η_ε** | **new** — fractional mismatch of the two gratings' *strain* coefficients | — | **1e-3** |
| ΔT_common / ΔT_diff | Common (whole-package) / differential temperature | — | — |
| τ | Averaging / measurement time (window ≈ 60 s) | 60 s | 60 s |
| f_u | Servo (loop) unity-gain bandwidth | — | — |

## 2. What changes when the ring becomes a grating

The ratiometric argument is untouched: Δν = S·(T_B − T_A), the wanted quantity is a material-defined ratio over a 60 s window, so S cancels, slow common terms cancel, and the figure of merit is σ_Δν(τ = 60 s). What changes is **which contributors are big**.

### 2.1 The one big win — linewidth
The microring notebook's central concession was that *"true dispersive PDH is unreachable on a Q = 1e5 ≈ 1.9 GHz line"*, which forced the dither scheme and its carrier-smear penalty. A commercial **π-phase-shifted FBG has a transmission notch of 15–50 MHz** (TeraXion parts are specified at 15 MHz FWHM, i.e. Q ≈ 1.3e7), and π-FBGs have been PDH-locked directly in the literature. So:

- the discriminant slope goes as **1/FWHM**, so every "fraction of a linewidth" error — detection noise, RIN pulling, RAM lock offset — scales down by **FWHM_FBG / FWHM_ring ≈ 50 MHz / 1.9 GHz ≈ 0.026**, a **38× improvement**. This gain is available to *any* slope discriminator, not only to textbook PDH, provided the modulation depth δν is scaled down with the linewidth — see §11.
- **the carrier-smear residual goes to zero**, because the modulation depth needed on a 50 MHz line is small enough that the carrier is no longer being dragged across an appreciable fraction of the resonance.

> ⚠ **Correction — what "PDH" means in this notebook.** An earlier draft claimed the dither scheme is retired and true dispersive PDH becomes available. **That is not true on the v7 electronics.** True dispersive PDH needs Ω ≳ FWHM, i.e. ≥50 MHz of phase modulation; the v7 hardware modulates the *laser current* through a bias-tee at **Ω = 100 kHz–3.5 MHz** (2–2.5 MHz nominal) from an AD9833 DDS on a 25 MHz clock. So the FBG is locked with **the same 2–2.5 MHz derivative / lock-in discriminator the microring uses** — which is exactly what the microring notebook's "RF" column already models. The column labelled **"π-FBG PDH"** throughout this notebook therefore means *"the v7 derivative lock, on a 50 MHz line"*, and its numbers are derived by scaling the ring's RF column. The linewidth gain is real and survives; only the *name* was wrong. §11 works through what would have to change for true PDH.

- **residual laser FM does *not* improve.** Residual FM is the laser's free-running noise above the loop unity-gain bandwidth f_u — a property of the DFB and of f_u, not of the reference linewidth. A narrower line buys error-signal amplitude, which *could* buy loop gain and a higher f_u, but on the v7 cards f_u is pinned by the AFE post-mix pole (~72 kHz) and by Ω. The parameter `fm_improves_with_linewidth` defaults to **False** for this reason; set it True only if the loop bandwidth is actually raised.

### 2.2 The one big loss — sensitivity
S falls from 2 GHz/K to **1.25 GHz/K**. The same frequency uncertainty is therefore **1.6× more kelvin**. This partly eats the linewidth win and must be carried explicitly — it is why the FBG budget is quoted in Hz first and converted second.

### 2.3 Three genuinely new terms with no microring analogue

| # | Term | Why it is new | Cancels in the ratio? |
|---|---|---|---|
| 1 | **Fiber thermodynamic (thermoconductive) noise** | The reference is now a ~10 mm length of silica fiber, not a lithographic ring on a die. Spontaneous temperature fluctuations of the grating volume are the *fundamental* floor of every FBG frequency reference and were measured directly in FBG lasers. Over a 60 s window the two gratings are separated by far more than the thermal diffusion length in silica (√(Dτ) ≈ 7 mm at 60 s, D ≈ 0.85 mm²/s) — which the thermal barrier *requires* — so the two gratings' fluctuations are **uncorrelated** and add in quadrature. | **No.** This is the FBG's hard floor. |
| 2 | **Parasitic-etalon / interferometric fringe** | The optical path is now metres of fiber with connectors and splices. Any residual reflection pair forms a weak etalon; a 1 m parasitic cavity has an FSR of ~100 MHz, *comparable to the π-FBG linewidth itself*, so the fringe appears as a first-order tilt on the PDH baseline rather than an averaging ripple. Its phase drifts with fiber temperature. **This is the FBG's version of "measure the RAM spectrum early."** Mitigation: all-spliced path, 8° APC everywhere a connector is unavoidable, isolator after the laser. | **No** (flicker part). |
| 3 | **Polarisation / grating birefringence** | Fiber birefringence splits the Bragg resonance into two peaks. In non-PM fiber the launch polarisation wanders and drags the lock point between them. PM fiber with a properly aligned launch reduces this by ~15×; it does not remove it (PER is finite). | **No.** |

### 2.4 One term that gets much worse, and one that saves it
The FBG's strain sensitivity, **S_ε = −150 MHz/µε**, is enormous next to its thermal sensitivity: **1 nε ≡ 0.12 mK-equivalent**. Reaching nanokelvin-equivalent stability naively demands ~10 fε of strain stability, which is not achievable in a packaged fiber.

What was *supposed* to save it is topology: two gratings in series in one fiber between common anchors share axial strain identically, rejecting it to the coefficient mismatch η_ε ≈ 1e-3.

> ⚠ **Correction — the two gratings cannot share one fiber.** Section 2.5 requires the pair to be matched to **±10 pm** so the beat lands near 100 MHz. Two gratings that close in wavelength have almost completely overlapping stop bands, so in a series fiber each laser would be reflected by the *other* grating and could not address its own. The two lasers must therefore run in **two separate fibers, one grating each**, bonded to a common substrate across a thermal barrier — the direct analogue of "two rings on one die", and a clean match to the AFE's two independent PD/TIA/demod channels.
>
> **Consequence for this budget:** common-strain rejection is no longer a *material* property of a shared fiber. It becomes a **mounting specification** — symmetric bond geometry, matched adhesive, matched bond length, common anchors on one substrate. η_ε = 1e-3 is therefore an *assumption about the mount*, not about the glass, and it is the single least-defended number in this notebook. It should be measured on the assembled head, not assumed.

With that caveat, the budget splits strain into:
- **common axial strain × η_ε** — slow, random-walk, **cancels**;
- **differential strain** from coating creep, adhesive relaxation, local bend and twist between the two gratings — a random-walk part that **cancels** and a **flicker part that does not**. The flicker part is a real, non-negotiable packaging term.

### 2.5 A practical constraint the ring did not have — pair matching
Catalogue FBG centre-wavelength tolerance is **±0.1 to ±0.2 nm**, i.e. **±12 to ±25 GHz** of Bragg offset. The nominal beat is 100 MHz. Even a *matched pair* specified to ±10 pm leaves a worst-case **2.5 GHz** offset. **The two gratings must be trimmed into beat range**, and Section 2b sizes that trim two ways:

- **TEC trim — unusable.** Closing a catalogue pair takes a **~20 K** offset between the two gratings. That offset *is* the measurand; it destroys the differential thermometer. Even a matched pair needs 2 K.
- **Strain trim — cheap.** Because S_ε is so large, a matched pair closes with **~17 µε ≈ 0.17 µm** of stretch over a 10 mm grating: one PZT stack on one grating. A catalogue pair takes ~170 µε (1.7 µm), which is mechanically feasible but leaves a large standing strain bias that will creep — exactly the term Section 2.4 says not to feed.

Order **matched pairs from one draw and one phase mask**, and budget a strain trim. Do not try to do it with the TECs.

### 2.6 What is unchanged
Detection chain, DDS/LO, counter and timebase, the servo topology, the ratiometric framing, and the 60 s window. The v7 CH/AFE/DIG card architecture drives an FBG head exactly as it drives a ring head; the fiber head replaces the die, and the EOM (needed for real PDH) is the one added optical component.

In [ ]:
import numpy as np
import pandas as pd
C = 299_792_458.0  # m/s

In [ ]:
from dataclasses import dataclass, field

@dataclass
class FBGParams:
    """Commercially-available fiber-Bragg-grating differential thermometer.

    Two gratings written in one PM fiber, separated by a thermal barrier.
    Scheme 'pdh'  -> pi-phase-shifted FBG, true dispersive PDH.
    Scheme 'sof'  -> standard telecom FBG, side-of-fringe intensity lock.
    """
    # --- optical / grating ---------------------------------------------------
    lam: float            = 1550e-9
    n_eff: float          = 1.4468        # SMF-28 at 1550 nm
    L_g: float            = 10e-3         # grating length [m]

    fwhm_pdh: float       = 50e6          # pi-FBG notch FWHM [Hz]  (TeraXion spec 15 MHz)
    fwhm_sof: float       = 25e9          # standard FBG 0.2 nm reflection FWHM [Hz]
    fwhm_ring: float      = 1.9e9         # microring reference point, for scaling

    # --- sensitivities (silica FBG at 1550 nm) -------------------------------
    dlam_dT: float        = 10e-12        # 10 pm/K    -> S     = 1.25 GHz/K
    dlam_deps_ue: float   = 1.2e-12       # 1.2 pm per MICROstrain -> 150 MHz/ue

    # --- beat / readout / window ---------------------------------------------
    Delta_nu: float       = 100e6         # nominal beat [Hz]
    fs_span: float        = 20e9          # full-scale measurand range of Delta_nu [Hz]
    tau: float            = 1.0           # snapshot averaging time [s]
    tau_meas: float       = 60.0          # measurement window [s]  <-- the spec point
    tau_max: float        = 600.0         # Allan deviation extent [s]
    ratio_cancel: bool    = True

    # --- pair matching --------------------------------------------------------
    lam_tol_catalogue: float = 100e-12    # +/-0.1 nm typical catalogue tolerance
    lam_tol_matched: float   = 10e-12     # +/-10 pm matched-pair spec

    # --- detection white-FM sqrt-PSD, referred to the RING, [Hz/sqrt(Hz)] -----
    # scaled to the FBG by the linewidth ratio inside the model
    det_psd_ring: float   = 20.0

    # --- coefficient matching -------------------------------------------------
    eps: float            = 5.0e-4   # dnu/dT match, same fiber + same phase mask
    eta_eps: float        = 1.0e-3   # dnu/deps match
    dT_common_rep: float  = 10e-6
    dT_common_repro: float= 1.0e-3
    dT_diff_rep: float    = 10e-9
    dT_diff_repro: float  = 1.0e-6
    eps_common_rep: float = 1.0e-9   # common axial strain [strain], repeatability
    eps_common_repro: float = 1.0e-7

    # --- timebase fractional stability ---------------------------------------
    frac_pdh: float       = 1e-8
    frac_sof: float       = 1e-8

    # === THE THREE FBG-SPECIFIC INPUTS (measure these early) =================
    # 1) fiber thermodynamic noise in one grating volume, Hz @1s, flicker.
    #    Two uncorrelated gratings -> combined = sqrt(2) * this.  DOES NOT CANCEL.
    fiber_thermal_1g: float   = 7.0
    # 2) parasitic-etalon fringe, Hz @1s. Flicker part survives, drift cancels.
    etalon_flicker_pdh: float = 5.0
    etalon_flicker_sof: float = 5.0
    etalon_drift: float       = 60.0
    # 3) polarisation / birefringence lock wander, Hz @1s, flicker.
    pol_pm: float             = 2.0    # PM fiber, aligned launch
    pol_smf: float            = 30.0   # non-PM fallback, for comparison
    use_pm_fiber: bool        = True

    # --- EOM residual amplitude modulation (referred to the ring) -------------
    ram_flicker_ring: float   = 20.0
    ram_drift_ring: float     = 50.0

    # --- laser / intensity (referred to the ring) -----------------------------
    laser_fm_ring: float      = 6.0
    # Residual laser FM above the loop unity-gain BW f_u is a property of the LASER
    # and f_u -- NOT of the reference linewidth. It improves only if the narrower
    # line is actually used to raise f_u. The v7 electronics cannot: f_u is pinned
    # by the AFE post-mix pole (~72 kHz) and Omega = 2-2.5 MHz. Default False.
    fm_improves_with_linewidth: bool = False
    rin_ring: float           = 5.0
    # side-of-fringe has no RIN rejection; assume power normalisation buys 10x back
    sof_rin_penalty: float    = 10.0

    # --- strain / packaging ---------------------------------------------------
    strain_diff_flicker: float = 3.0    # differential coating/adhesive creep, KEPT
    strain_diff_rw: float      = 40.0   # random-walk part, cancels
    strain_common_rw: float    = 20.0   # common axial x eta_eps, cancels

    k: float = 2.0

    # --- derived --------------------------------------------------------------
    @property
    def nu0(self):    return C / self.lam
    @property
    def dnu_dlam(self): return C / self.lam**2        # Hz per metre of wavelength
    @property
    def S(self):      return self.dnu_dlam * self.dlam_dT           # Hz/K
    @property
    def S_eps_ue(self):
        """Hz per MICROstrain -- the number people quote (150 MHz/ue)."""
        return self.dnu_dlam * self.dlam_deps_ue
    @property
    def S_eps(self):
        """Hz per UNIT strain -- the number the arithmetic needs (1.5e14 Hz)."""
        return self.S_eps_ue * 1e6
    @property
    def Q_pdh(self):  return self.nu0 / self.fwhm_pdh
    @property
    def Q_sof(self):  return self.nu0 / self.fwhm_sof
    def fwhm(self, scheme):  return self.fwhm_pdh if scheme == "pdh" else self.fwhm_sof
    def lw_scale(self, scheme):
        """Discriminant scaling vs the microring: slope goes as 1/FWHM."""
        return self.fwhm(scheme) / self.fwhm_ring
    @property
    def pol(self):    return self.pol_pm if self.use_pm_fiber else self.pol_smf

### 2a. Device constants — what the catalogue parts actually give you

In [ ]:
p = FBGParams()
rows = [
    ["Carrier nu_0",                     f"{p.nu0/1e12:.3f} THz"],
    ["dnu/dlam at 1550 nm",              f"{p.dnu_dlam*1e-9*1e-9:.1f} GHz/nm"],
    ["S = dnu/dT  (10 pm/K)",            f"{p.S/1e9:.2f} GHz/K      (ring: 2.00 GHz/K)"],
    ["S_eps = dnu/deps  (1.2 pm/ue)",    f"{p.S_eps_ue/1e6:.0f} MHz/ue  = {p.S_eps:.2e} Hz per unit strain"],
    ["1 nano-strain in kelvin-equivalent", f"{p.S_eps*1e-9/p.S*1e3:.2f} mK"],
    ["1 nK in strain-equivalent",        f"{p.S*1e-9/p.S_eps*1e15:.1f} femtostrain"],
    ["pi-FBG FWHM",                      f"{p.fwhm_pdh/1e6:.0f} MHz   -> Q = {p.Q_pdh:.2e}"],
    ["standard FBG FWHM",                f"{p.fwhm_sof/1e9:.0f} GHz    -> Q = {p.Q_sof:.2e}"],
    ["microring FWHM (reference)",       f"{p.fwhm_ring/1e9:.2f} GHz  -> Q = 1.0e+05"],
    ["discriminant gain, pi-FBG vs ring", f"{1/p.lw_scale('pdh'):.0f}x better"],
    ["discriminant loss, std FBG vs ring", f"{p.lw_scale('sof'):.0f}x worse"],
    ["thermal diffusion length in silica @ 60 s", f"{np.sqrt(0.85e-6*60)*1e3:.1f} mm"],
]
display(pd.DataFrame(rows, columns=["Quantity", "Value"]).style.hide(axis="index"))

### 2b. Pair matching and the strain trim

The beat must land near Δν = 100 MHz for the counter to see it. Catalogue tolerance does not get close; even a matched pair does not. Size the trim.

In [ ]:
def pair_matching(p: FBGParams):
    rows = []
    for label, tol in [("catalogue +/-0.1 nm", p.lam_tol_catalogue),
                       ("matched pair +/-10 pm", p.lam_tol_matched)]:
        # worst-case pair offset is 2*tol (one high, one low)
        dnu = p.dnu_dlam * 2 * tol
        strain = dnu / p.S_eps                 # [unit strain]
        rows.append([label,
                     f"{2*tol*1e12:.0f} pm",
                     f"{dnu/1e9:.2f} GHz",
                     f"{dnu/p.S:.2f} K",                 # temperature trim needed
                     f"{strain*1e6:.1f} ue",             # strain trim needed
                     f"{strain*p.L_g*1e6:.2f} um"])      # stretch over L_g
    df = pd.DataFrame(rows, columns=["Pair spec", "Worst-case dlam", "Beat offset",
                                     "TEC trim needed", "Strain trim needed",
                                     f"Stretch over L_g={p.L_g*1e3:.0f} mm"])
    return df.style.hide(axis="index").set_properties(
        subset=list(df.columns[1:]), **{"text-align": "right"})

print("Trim required to pull a grating PAIR onto the nominal 100 MHz beat:")
display(pair_matching(p))
print("\nThe TEC route is unusable: closing a catalogue pair needs a ~20 K offset between the")
print("two gratings, which IS the measurand -- it destroys the differential thermometer.")
print("The strain route is cheap: a matched pair needs ~17 ue = 0.17 um of stretch over a")
print("10 mm grating, one PZT stack on one grating. Specify a matched pair AND a strain trim.")

## 3. Full contributor budget — snapshot (Hz, then Kelvin)

Kept for completeness and for parity with the microring notebook. The reproducibility column is **not** the spec for this ratiometric measurement (see Sections 4–5); it is shown to expose the slow terms that the ratio removes. The two columns are now **π-FBG / PDH** and **standard FBG / side-of-fringe** — the dither scheme is gone, because a π-FBG makes dispersive PDH reachable.

> **Read the strain row, not the total.** The snapshot combination below is **dominated by the common-axial-strain term** — ~150 Hz on repeatability and ~15 kHz on reproducibility, from an assumed 1 nε / 0.1 µε common excursion beating against the η_ε = 1e-3 coefficient match. That is not an error in the model; it is the whole point of Section 2.4. An FBG pair measured *open-loop*, without the ratio, is a strain gauge that happens to respond to temperature. Everything in this budget that makes the instrument work — the series topology, the common anchors, the ratio — exists to kill that row. Sections 4–5 are where it dies.

In [ ]:
def detection_diff(psd, tau):        return psd / np.sqrt(tau)
def thermal_gradient(dT, S):         return dT * S
def thermal_mismatch(eps, dTc, S):   return eps * dTc * S
def strain_mismatch(eta, de, Se):    return eta * de * Se
def timebase(frac, dv):              return frac * dv

def build_snapshot(p: FBGParams):
    S, Se = p.S, p.S_eps
    out = {}
    for scheme in ("pdh", "sof"):
        r = p.lw_scale(scheme)                       # discriminant scaling vs ring
        det = detection_diff(p.det_psd_ring * r, p.tau)
        rin = p.rin_ring * r * (p.sof_rin_penalty if scheme == "sof" else 1.0)
        ram = p.ram_flicker_ring * r if scheme == "pdh" else 0.0   # no EOM in SOF
        fm  = p.laser_fm_ring * (np.sqrt(r) if p.fm_improves_with_linewidth else 1.0)
        eta = p.etalon_flicker_pdh if scheme == "pdh" else p.etalon_flicker_sof
        tb  = timebase(p.frac_pdh if scheme == "pdh" else p.frac_sof, p.Delta_nu)
        out[scheme] = dict(det=det, rin=rin, ram=ram, fm=fm, eta=eta, tb=tb)

    g1, g2 = thermal_gradient(p.dT_diff_rep, S),  thermal_gradient(p.dT_diff_repro, S)
    m1, m2 = thermal_mismatch(p.eps, p.dT_common_rep, S), thermal_mismatch(p.eps, p.dT_common_repro, S)
    s1, s2 = strain_mismatch(p.eta_eps, p.eps_common_rep, Se), strain_mismatch(p.eta_eps, p.eps_common_repro, Se)
    ft = p.fiber_thermal_1g * np.sqrt(2.0)          # two uncorrelated gratings
    pol = p.pol
    a, b = out["pdh"], out["sof"]

    rows = [
        ("Detection / discrimination noise",  "A . mod",  (a["det"], a["det"], b["det"], b["det"])),
        ("Residual laser FM above loop BW",   "A . mod",  (a["fm"],  a["fm"],  b["fm"],  b["fm"])),
        ("RIN -> lock-point pulling",         "B . mod",  (a["rin"], 2*a["rin"], b["rin"], 2*b["rin"])),
        ("EOM RAM lock offset",               "B . mod",  (a["ram"], 5*a["ram"], b["ram"], b["ram"])),
        ("Fiber thermodynamic noise (x sqrt2)","A . ref", (ft, ft, ft, ft)),
        ("Parasitic etalon / fringe drift",   "B . ref",  (a["eta"], 6*a["eta"], b["eta"], 6*b["eta"])),
        ("Polarisation / birefringence",      "B . ref",  (pol, 3*pol, pol, 3*pol)),
        ("Thermal - gradient (dT_diff . S)",  "B . env",  (g1, g2, g1, g2)),
        ("Thermal - mismatch (eps.S.dTc)",    "B . env",  (m1, m2, m1, m2)),
        ("Strain - common x eta_eps",         "B . env",  (s1, s2, s1, s2)),
        ("Strain - differential (creep)",     "B . env",  (p.strain_diff_flicker, p.strain_diff_rw,
                                                           p.strain_diff_flicker, p.strain_diff_rw)),
        ("Timebase / counter reference",      "B . read", (a["tb"], a["tb"], b["tb"], b["tb"])),
    ]
    data = np.array([r[2] for r in rows], float)
    uc = np.sqrt((data**2).sum(0)); U = p.k*uc
    cols = ["pi-FBG PDH - repeat.", "pi-FBG PDH - reprod.",
            "std FBG SOF - repeat.", "std FBG SOF - reprod."]
    return dict(rows=rows, data=data, uc=uc, U=U, cols=cols, S=S, nu0=p.nu0,
                frac_uc=uc/p.nu0, frac_U=U/p.nu0, fs_uc=uc/p.fs_span, fs_U=U/p.fs_span,
                bits=np.log2(p.fs_span/uc))

In [ ]:
def fmt_hz(v):
    if isinstance(v, str): return v
    if v == 0: return "\u2014"
    if abs(v) < 1: return f"{v:.3g}"
    return f"{v:,.1f}"

def fmt_K(v, S):
    if isinstance(v, str): return v
    if v == 0: return "\u2014"
    kk = v / S; a = abs(kk)
    if a < 1e-6: return f"{kk*1e9:.1f} nK"
    if a < 1e-3: return f"{kk*1e6:.3g} \u00b5K"
    return f"{kk*1e3:.3g} mK"

def _style(disp, num, summary_mask):
    sty = disp.style.hide(axis="index").set_properties(subset=num, **{"text-align": "right"})
    sty = sty.apply(lambda r: ["font-weight: 700" if summary_mask[r.name] else "" for _ in r], axis=1)
    return sty

def snapshot_table_hz(info, k=2):
    cols = info["cols"]; ef = lambda x: f"{x:.2e}"
    rows = [[n, t]+list(v) for (n,t,v) in info["rows"]]
    rows += [["Combined standard uncertainty u_c (k=1)", ""]+list(info["uc"]),
             [f"Expanded uncertainty U (k={k:g}, ~95%)", ""]+list(info["U"]),
             ["Fractional u_c  (\u0394\u03bd/\u03bd\u2080, carrier)", ""]+[ef(x) for x in info["frac_uc"]],
             ["Fractional u_c  (\u0394\u03bd/FS)", ""]+[ef(x) for x in info["fs_uc"]],
             ["Effective resolution over FS [bits]", ""]+[f"{b:.1f}" for b in info["bits"]]]
    df = pd.DataFrame(rows, columns=["Contributor","Type"]+cols)
    summ = df["Type"].astype(str).str.strip().eq("").tolist()
    disp = df.copy()
    for c in cols: disp[c] = disp[c].map(fmt_hz)
    return _style(disp, cols, summ)

def snapshot_table_K(info, k=2):
    S = info["S"]; cols = info["cols"]
    rows = [[n, t]+[x/S for x in v] for (n,t,v) in info["rows"]]
    rows += [["Combined standard uncertainty u_c (k=1)", ""]+list(info["uc"]/S),
             [f"Expanded uncertainty U (k={k:g}, ~95%)", ""]+list(info["U"]/S)]
    df = pd.DataFrame(rows, columns=["Contributor","Type"]+cols)
    summ = df["Type"].astype(str).str.strip().eq("").tolist()
    disp = df.copy()
    for c in cols: disp[c] = disp[c].map(lambda v: v if isinstance(v,str) else fmt_K(v*S, S))
    return _style(disp, cols, summ)

p = FBGParams()
snap = build_snapshot(p)
print(f"nu0={snap['nu0']:.4e} Hz   S={p.S:.3e} Hz/K   S_eps={p.S_eps:.3e} Hz/strain")
print(f"pi-FBG FWHM={p.fwhm_pdh/1e6:.0f} MHz (Q={p.Q_pdh:.1e})   std FBG FWHM={p.fwhm_sof/1e9:.0f} GHz (Q={p.Q_sof:.1e})")
snapshot_table_hz(snap)

Kelvin twin of the snapshot budget (each Hz value ÷ S, with S = 1.25 GHz/K — note this is a **1.6× worse** Hz→K conversion than the ring's 2 GHz/K):

In [ ]:
snapshot_table_K(snap)

## 4. Ratiometric within-window model (the operative budget)

Each contributor gets an Allan power-law slope μ (σ ∝ τ^μ: white-FM −½, flicker-FM 0, random-walk-FM +½) and a flag for whether the **ratio cancels** it. The combined within-window value is evaluated at **τ_meas = 60 s** over the surviving terms only.

Where the microring notebook had one decisive unknown (in-band RAM), the FBG version has **three**, and they are the three new terms of Section 2.3:

1. **fiber thermodynamic noise** — non-cancelling flicker, ×√2 for two uncorrelated gratings. Expected to be the floor.
2. **parasitic-etalon fringe** — flicker part non-cancelling, drift part cancels.
3. **polarisation wander** — non-cancelling flicker; PM fiber or 15× worse.

The differential-strain creep splits the same way: flicker survives, random-walk cancels.

In [ ]:
def allan_contributors(p: FBGParams, scheme):
    """(name, a@1s [Hz], mu, ratio_cancels) for one lock scheme."""
    r  = p.lw_scale(scheme)
    ft = p.fiber_thermal_1g * np.sqrt(2.0)
    rin = p.rin_ring * r * (p.sof_rin_penalty if scheme == "sof" else 1.0)
    ram = p.ram_flicker_ring * r if scheme == "pdh" else 0.0
    ramd= p.ram_drift_ring   * r if scheme == "pdh" else 0.0
    eta = p.etalon_flicker_pdh if scheme == "pdh" else p.etalon_flicker_sof
    tb  = (p.frac_pdh if scheme == "pdh" else p.frac_sof) * p.Delta_nu
    return [
        # name                              a@1s              mu    cancels
        ("Detection / discrimination",       p.det_psd_ring*r, -0.5, False),
        ("Residual laser FM floor",          p.laser_fm_ring*(np.sqrt(r) if p.fm_improves_with_linewidth else 1.0), 0.0, False),
        ("RIN in-band",                      rin,               0.0, False),
        ("EOM RAM in-band (flicker)",        ram,               0.0, False),
        ("Fiber thermodynamic noise",        ft,                0.0, False),
        ("Parasitic etalon (flicker)",       eta,               0.0, False),
        ("Polarisation / birefringence",     p.pol,             0.0, False),
        ("Strain - differential (flicker)",  p.strain_diff_flicker, 0.0, False),
        ("Timebase short-term",              tb,               -0.5, False),
        ("EOM RAM slow drift",               ramd,              0.5, True),
        ("Parasitic etalon slow drift",      p.etalon_drift,    0.5, True),
        ("Thermal common-mode leakage",      30.0,              0.5, True),
        ("Strain - common axial x eta_eps",  p.strain_common_rw,0.5, True),
        ("Strain - differential creep (RW)", p.strain_diff_rw,  0.5, True),
    ]

def sigma_components(p, tau, scheme):
    return {n: (a * tau**mu, mu, canc) for (n, a, mu, canc) in allan_contributors(p, scheme)}

def sigma_allan(p, taus, scheme, ratio=True):
    taus = np.atleast_1d(np.asarray(taus, float)); tot = np.zeros_like(taus)
    for (n, a, mu, canc) in allan_contributors(p, scheme):
        if ratio and canc: continue
        tot += (a * taus**mu)**2
    return np.sqrt(tot)

In [ ]:
def within_window_table(p: FBGParams, hz=True):
    tau = p.tau_meas; S = p.S
    cd = sigma_components(p, tau, "pdh")
    cr = sigma_components(p, tau, "sof")
    rows = []
    for (n, a, mu, canc) in allan_contributors(p, "pdh"):
        rows.append([n, "cancels" if canc else "kept", cd[n][0], cr[n][0]])
    cols_n = [f"pi-FBG PDH @ {tau:g}s", f"std FBG SOF @ {tau:g}s"]
    df = pd.DataFrame(rows, columns=["Contributor", "In ratio"] + cols_n)
    uc_d = sigma_allan(p, tau, "pdh", ratio=p.ratio_cancel)[0]
    uc_r = sigma_allan(p, tau, "sof", ratio=p.ratio_cancel)[0]
    df.loc[len(df)] = [f"Combined (ratio={p.ratio_cancel}) @ {tau:g}s", "", uc_d, uc_r]
    summ = (df["In ratio"].astype(str).str.strip().eq("")).tolist()
    disp = df.copy()
    fmt = (lambda v: fmt_hz(v)) if hz else (lambda v: fmt_K(v, S))
    for c in cols_n: disp[c] = disp[c].map(lambda v: v if isinstance(v,str) else fmt(v))
    sty = disp.style.hide(axis="index").set_properties(subset=cols_n, **{"text-align":"right"})
    def rowstyle(r):
        if summ[r.name]: return ["font-weight: 700" for _ in r]
        if str(df.iloc[r.name]["In ratio"]) == "cancels":
            return ["color: #999; text-decoration: line-through" for _ in r]
        return ["" for _ in r]
    return sty.apply(rowstyle, axis=1), (uc_d, uc_r)

p = FBGParams()
print("WITHIN-WINDOW BUDGET (Hz) at tau_meas =", p.tau_meas, "s")
sty_hz, (ucd, ucr) = within_window_table(p, hz=True)
print(f"Combined: pi-FBG PDH {ucd:.1f} Hz ({ucd/p.S*1e9:.1f} nK)   "
      f"std FBG SOF {ucr:.1f} Hz ({ucr/p.S*1e9:.0f} nK)")
sty_hz

Kelvin twin (within-window):

In [ ]:
within_window_table(p, hz=False)[0]

### 4b. σ at key averaging times (Hz and Kelvin)

In [ ]:
def sigma_vs_tau_table(p: FBGParams, taus=(1, 10, 60, 100, 600)):
    S = p.S
    rec = [(t, sigma_allan(p, t, "pdh", p.ratio_cancel)[0],
               sigma_allan(p, t, "sof", p.ratio_cancel)[0]) for t in taus]
    df = pd.DataFrame(rec, columns=["tau [s]", "pi-FBG PDH", "std FBG SOF"])
    df_hz = df.copy()
    for c in ["pi-FBG PDH", "std FBG SOF"]:
        df_hz[c] = df_hz[c].map(lambda v: f"{v:,.2f} Hz")
    df_K = df.copy()
    for c in ["pi-FBG PDH", "std FBG SOF"]:
        df_K[c] = df_K[c].map(lambda v: fmt_K(v, S))
    df_K = df_K.rename(columns={"pi-FBG PDH":"pi-FBG PDH (\u0394T)",
                                "std FBG SOF":"std FBG SOF (\u0394T)"})
    return df_hz, df_K

t_hz, t_K = sigma_vs_tau_table(p)
display(t_hz.style.hide(axis="index"))
display(t_K.style.hide(axis="index"))

### 4c. Head-to-head against the microring budget

Same window, same ratiometric framing, same signal chain — only the frequency reference changed.

In [ ]:
def head_to_head(p: FBGParams):
    S_ring = 2.0e9
    # microring RF-scheme contributors, transcribed from the microring notebook
    ring = [("Detection", 20.0, -0.5), ("Laser FM", 6.0, 0.0), ("RIN", 5.0, 0.0),
            ("RAM in-band", 20.0, 0.0), ("Timebase", 0.1, -0.5)]
    tau = p.tau_meas
    ring_uc      = np.sqrt(sum((a*tau**mu)**2 for _, a, mu in ring))
    ring_uc_best = np.sqrt(sum((a*tau**mu)**2 for n, a, mu in ring if n != "RAM in-band"))
    fbg_uc  = sigma_allan(p, tau, "pdh", p.ratio_cancel)[0]
    fbg_sof = sigma_allan(p, tau, "sof", p.ratio_cancel)[0]
    fbg_best = sigma_allan(FBGParams(fiber_thermal_1g=3.0, etalon_flicker_pdh=1.0),
                           tau, "pdh", True)[0]
    rows = [
        ["Microring, RF dither - in-band RAM survives", "2.00", "1.9 GHz",
         f"{ring_uc:.1f}", f"{ring_uc/S_ring*1e9:.1f} nK"],
        ["Microring, RF dither - RAM slower than window", "2.00", "1.9 GHz",
         f"{ring_uc_best:.1f}", f"{ring_uc_best/S_ring*1e9:.1f} nK"],
        ["pi-FBG, true PDH - nominal", f"{p.S/1e9:.2f}", f"{p.fwhm_pdh/1e6:.0f} MHz",
         f"{fbg_uc:.1f}", f"{fbg_uc/p.S*1e9:.1f} nK"],
        ["pi-FBG, true PDH - best case (low fiber thermal, etalon killed)",
         f"{p.S/1e9:.2f}", f"{p.fwhm_pdh/1e6:.0f} MHz",
         f"{fbg_best:.1f}", f"{fbg_best/p.S*1e9:.1f} nK"],
        ["Standard telecom FBG, side-of-fringe", f"{p.S/1e9:.2f}", f"{p.fwhm_sof/1e9:.0f} GHz",
         f"{fbg_sof:.1f}", f"{fbg_sof/p.S*1e9:.0f} nK"],
    ]
    df = pd.DataFrame(rows, columns=["Reference / scheme", "S [GHz/K]", "FWHM",
                                     f"u_c @ {tau:g}s [Hz]", f"u_c @ {tau:g}s [\u0394T]"])
    return df.style.hide(axis="index").set_properties(
        subset=list(df.columns[1:]), **{"text-align": "right"})

head_to_head(FBGParams())

## 5. Allan deviation out to 600 s

Solid = ratiometric (slow common terms cancelled); dashed = without ratio cancellation, showing the slow terms — etalon drift, common strain, thermal leakage — re-entering as a random-walk upturn. The 60 s measurement point is marked. Right axis is the equivalent differential temperature at S = 1.25 GHz/K.

Note the shape difference from the microring plot: the π-FBG curve is **flat** across the whole decade, because its surviving budget is almost entirely flicker (fiber thermal + etalon + polarisation). There is no averaging benefit past a few seconds — a longer window buys nothing, which is worth knowing before anyone proposes lengthening it.

In [ ]:
try:
    import matplotlib.pyplot as plt
    p = FBGParams()
    taus = np.logspace(0, np.log10(p.tau_max), 250)
    d_r = sigma_allan(p, taus, "pdh", True)
    s_r = sigma_allan(p, taus, "sof", True)
    d_n = sigma_allan(p, taus, "pdh", False)
    s_n = sigma_allan(p, taus, "sof", False)
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    ax.loglog(taus, d_r, label="pi-FBG PDH (ratiometric)")
    ax.loglog(taus, s_r, label="std FBG side-of-fringe (ratiometric)")
    ax.loglog(taus, d_n, "--", alpha=0.5, label="pi-FBG PDH (no ratio)")
    ax.loglog(taus, s_n, "--", alpha=0.5, label="std FBG SOF (no ratio)")
    ax.axvline(p.tau_meas, color="gray", ls=":", lw=1)
    ax.text(p.tau_meas*1.05, ax.get_ylim()[1]*0.5, f"{p.tau_meas:g} s window",
            color="gray", rotation=90, va="top")
    ax.set_xlabel("Averaging time \u03c4  [s]")
    ax.set_ylabel("Allan deviation \u03c3_\u0394\u03bd  [Hz]")
    ax.set_title("FBG beat-frequency Allan deviation \u2014 flicker-limited, no averaging benefit")
    ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=8)
    ax2 = ax.twinx(); ax2.set_yscale("log")
    ax2.set_ylim(np.array(ax.get_ylim())/p.S*1e9)   # nK
    ax2.set_ylabel("equivalent \u03c3_\u0394T  [nK]")
    plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed - skip the plot.")

### 5b. Which of the three unknowns actually decides the answer

The microring notebook's advice was *"measure the RAM spectrum early."* The FBG equivalent is a three-way sweep. Each row holds the other two at nominal and moves one.

In [ ]:
def sensitivity_sweep(p0: FBGParams):
    tau = p0.tau_meas
    base = sigma_allan(p0, tau, "pdh", True)[0]
    rows = [["nominal", "-", f"{base:.2f}", f"{base/p0.S*1e9:.1f} nK", "-"]]
    sweeps = [
        ("Fiber thermodynamic noise [Hz@1s, per grating]", "fiber_thermal_1g", [2.0, 5.0, 7.0, 15.0, 30.0]),
        ("Parasitic etalon flicker [Hz@1s]",               "etalon_flicker_pdh", [0.5, 2.0, 5.0, 15.0, 40.0]),
        ("Polarisation wander [Hz@1s]",                    "pol_pm",            [0.5, 2.0, 5.0, 15.0, 30.0]),
        ("Differential strain creep, flicker [Hz@1s]",     "strain_diff_flicker",[0.5, 3.0, 10.0, 30.0]),
    ]
    for label, field_, vals in sweeps:
        for v in vals:
            pp = FBGParams(**{**p0.__dict__, field_: v})
            u = sigma_allan(pp, tau, "pdh", True)[0]
            rows.append([label, f"{v:g}", f"{u:.2f}", f"{u/pp.S*1e9:.1f} nK",
                         f"{u/base:.2f}x"])
    df = pd.DataFrame(rows, columns=["Input swept", "Value", f"u_c @ {tau:g}s [Hz]",
                                     f"u_c [\u0394T]", "vs nominal"])
    return df.style.hide(axis="index").set_properties(
        subset=list(df.columns[1:]), **{"text-align": "right"})

sensitivity_sweep(FBGParams())

## 6. Interactive sliders (Colab-ready)

Enables Colab's widget manager; installs ipywidgets only if missing. The three FBG unknowns are the top three sliders. Untick **PM fiber** to see what non-PM fiber costs.

In [ ]:
try:
    from google.colab import output as _co
    _co.enable_custom_widget_manager()
except Exception:
    pass
try:
    import ipywidgets as W
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])
    import ipywidgets as W
from IPython.display import display

def interactive(fiber_thermal_1g=7.0, etalon_flicker_pdh=5.0, pol_pm=2.0,
                fwhm_pdh_MHz=50.0, strain_diff_flicker=3.0,
                use_pm_fiber=True, tau_meas=60.0, ratio_cancel=True):
    p = FBGParams(fiber_thermal_1g=fiber_thermal_1g,
                  etalon_flicker_pdh=etalon_flicker_pdh,
                  pol_pm=pol_pm, fwhm_pdh=fwhm_pdh_MHz*1e6,
                  strain_diff_flicker=strain_diff_flicker,
                  use_pm_fiber=use_pm_fiber,
                  tau_meas=tau_meas, ratio_cancel=ratio_cancel)
    sty, (ucd, ucr) = within_window_table(p, hz=True)
    print(f"@ {tau_meas:g}s  |  pi-FBG PDH {ucd:7.2f} Hz = {ucd/p.S*1e9:6.1f} nK    "
          f"std FBG SOF {ucr:8.1f} Hz = {ucr/p.S*1e9:7.0f} nK   (ratio={ratio_cancel})")
    display(sty)

W.interact(interactive,
           fiber_thermal_1g   =W.FloatSlider(value=7,  min=0, max=40, step=0.5, description="fiber therm"),
           etalon_flicker_pdh =W.FloatSlider(value=5,  min=0, max=40, step=0.5, description="etalon"),
           pol_pm             =W.FloatSlider(value=2,  min=0, max=40, step=0.5, description="polarisation"),
           fwhm_pdh_MHz       =W.FloatSlider(value=50, min=10, max=500, step=5, description="pi-FBG FWHM"),
           strain_diff_flicker=W.FloatSlider(value=3,  min=0, max=30, step=0.5, description="diff strain"),
           use_pm_fiber       =W.Checkbox(value=True,  description="PM fiber"),
           tau_meas           =W.FloatLogSlider(value=60, base=10, min=0, max=2.78, description="tau [s]"),
           ratio_cancel       =W.Checkbox(value=True,  description="ratio cancel"));

### 6b. No-widget what-if (always works)

In [ ]:
def recompute(**ov):
    p = FBGParams(**ov)
    sty, (ucd, ucr) = within_window_table(p, hz=True)
    print(f"@ {p.tau_meas:g}s  pi-FBG PDH {ucd:.2f} Hz = {ucd/p.S*1e9:.1f} nK   "
          f"std FBG SOF {ucr:.1f} Hz = {ucr/p.S*1e9:.0f} nK")
    return sty

# best realistic case: 15 MHz TeraXion part, all-spliced APC path, quiet fiber
recompute(fwhm_pdh=15e6, fiber_thermal_1g=3.0, etalon_flicker_pdh=1.0)

## 7. Clean tables (final) — Hz and Kelvin, with Markdown export

Within-window budget at τ = 60 s (the operative result), as styled tables and as Markdown for the project docs.

In [ ]:
def to_markdown(df_disp, num_cols):
    cols = list(df_disp.columns)
    sep = "|" + "|".join([" ---: " if c in num_cols else " --- " for c in cols]) + "|"
    out = ["| " + " | ".join(cols) + " |", sep]
    for _, row in df_disp.iterrows():
        out.append("| " + " | ".join(str(row[c]) for c in cols) + " |")
    return "\n".join(out)

p = FBGParams()
print("=== FBG within-window budget @ 60 s \u2014 Hertz ===")
display(within_window_table(p, hz=True)[0])
print("=== FBG within-window budget @ 60 s \u2014 Kelvin ===")
display(within_window_table(p, hz=False)[0])

t_hz, t_K = sigma_vs_tau_table(p)
print("\n\u03c3_\u0394\u03bd vs \u03c4 (Hz):\n");  print(to_markdown(t_hz, ["pi-FBG PDH","std FBG SOF"]))
print("\n\u03c3_\u0394T vs \u03c4 (Kelvin):\n")
print(to_markdown(t_K, ["pi-FBG PDH (\u0394T)","std FBG SOF (\u0394T)"]))

---
*Frequency result: with a **commercially available π-phase-shifted FBG** and true dispersive PDH, the ratiometric within-window uncertainty at τ ≈ 60 s lands in the **high-single-digit nanokelvin** range — comparable to, and in the nominal case slightly better than, the microring's RAM-limited figure, despite the FBG's 1.6× weaker S. The narrower line (50 MHz vs 1.9 GHz) buys ~38× on every lineshape-referred term and retires the dither scheme entirely.*

*But the floor moves. The microring was RAM-limited; **the FBG is limited by fiber thermodynamic noise and by parasitic-etalon fringes**, neither of which the ratio removes and neither of which averages down (both are flicker — see the flat Allan curve). The three things to measure early are, in order: **(1) the fiber thermal-noise floor of the actual grating pair, (2) the parasitic-fringe amplitude of the assembled fiber path, (3) the polarisation stability.***

*A **standard catalogue telecom FBG is not a substitute** — at 0.2 nm FWHM with side-of-fringe locking it is ~50× worse and lands in the tens-of-nanokelvin-to-sub-microkelvin range. If the FBG route is taken, the π-phase-shifted part is the design, not an upgrade.*

---

## 8. Voltage twin — square-law sensor readout (µV/V)

Used as a **voltmeter**, the response is still **square law**: ν = k₁·V². The microring calibration was **14 GHz of beat change over a 2 V full scale**. That number is a property of the *transducer chain*, and the chain ends at dν/dT — so with a thermally-coupled transducer the FBG's full-scale beat swing scales with S:

$$\nu_{FS}^{FBG} \;=\; \nu_{FS}^{ring}\cdot\frac{S_{FBG}}{S_{ring}} \;=\; 14\ \text{GHz}\times\frac{1.25}{2.00} \;=\; 8.75\ \text{GHz}$$

As before, what converts an uncertainty is the *slope*, not the chord:

$$\frac{d\nu}{dV} = 2k_1V = \frac{2\nu}{V}
\qquad\Longrightarrow\qquad
\frac{\sigma_V}{V} = \frac{1}{2}\,\frac{\sigma_\nu}{\nu}$$

**A result worth noting:** because ν_FS and S scale together, the **kelvin→volt conversion is invariant** — 1 nK is 143 pV for the FBG exactly as it was for the ring. What is *not* invariant is the **hertz→volt** conversion, which is **1.6× worse** (8.7 GHz/V of slope instead of 14 GHz/V). So compare the two references in kelvin, not in hertz.

> **Three caveats.**
> 1. A square-law sensor has **no single-number spec**. Both figures degrade below full scale, at different rates: σ_V = σ_ν/(2k₁V) goes as **1/V**, and σ_V/V = σ_ν/(2k₁V²) goes as **1/V²** (penalty table below). Every voltage number is quoted **at V_FS = 2 V**; never carry one to another operating point without rescaling.
> 2. `fs_span = 20e9` in `FBGParams` predates this calibration; against a real 8.7 GHz span the "Δν/FS" and effective-bits rows of Section 3 are optimistic by 20/8.7 ≈ 2.3×.
> 3. The ν_FS scaling above assumes the transducer couples to the reference **thermally**. If it couples through **strain** instead, use S_ε = 150 MHz/µε and recalibrate — set `NU_FS` directly rather than scaling it. An FBG is ~120× more strain-sensitive per kelvin-equivalent than it is temperature-sensitive, so a strain-coupled design changes this section completely.

In [ ]:
# --- square-law voltage readout, FBG ------------------------------------------
p = FBGParams()
S_RING   = 2.0e9
NU_FS_RING = 14.0e9
NU_FS = NU_FS_RING * (p.S / S_RING)   # thermally-coupled transducer -> scales with S
V_FS  = 2.0
K1    = NU_FS / V_FS**2               # square-law coefficient [Hz/V^2]

def dnu_dV(V=V_FS):             return 2.0 * K1 * V
def hz_to_volt(sig_hz, V=V_FS): return sig_hz / dnu_dV(V)
def hz_to_uVV(sig_hz, V=V_FS):  return sig_hz / (dnu_dV(V) * V) * 1e6
def K_to_uVV(sig_K, S, V=V_FS): return hz_to_uVV(sig_K * S, V)

def fmt_V(v, V=V_FS):
    if isinstance(v, str): return v
    if v == 0: return "\u2014"
    x = hz_to_volt(v, V); a = abs(x)
    if a < 1e-9:  return f"{x*1e12:.3g} pV"
    if a < 1e-6:  return f"{x*1e9:.3g} nV"
    return f"{x*1e6:.3g} \u00b5V"

print(f"nu_FS = {NU_FS/1e9:.2f} GHz over {V_FS:g} V  (ring: {NU_FS_RING/1e9:.0f} GHz)")
print(f"k1 = {K1/1e9:.2f} GHz/V^2 | slope at FS = {dnu_dV()/1e9:.2f} GHz/V "
      f"(chord {NU_FS/V_FS/1e9:.2f} GHz/V)")
print(f"1 Hz  -> {hz_to_volt(1)*1e12:.1f} pV = {hz_to_uVV(1):.3e} uV/V   "
      f"(ring: 71.4 pV -- FBG is 1.6x worse per Hz)")
print(f"1 nK  -> {p.S*1e-9:.2f} Hz -> {hz_to_volt(p.S*1e-9)*1e12:.0f} pV "
      f"= {K_to_uVV(1e-9, p.S):.3e} uV/V   (ring: 143 pV -- INVARIANT)")

### 8a. Within-window budget at τ = 60 s — combined and expanded, in µV/V

`u_c` is the ratiometric combination of the surviving terms (Section 4); `U = k·u_c` with k = 2 (≈95 %). Relative columns are quoted **at full scale (2 V)**.

In [ ]:
def voltage_summary(p: FBGParams, cases=None, k=None):
    k = p.k if k is None else k
    tau = p.tau_meas
    if cases is None:
        cases = [
            ("pi-FBG PDH  \u2014 nominal", FBGParams(**{**p.__dict__}), "pdh"),
            ("pi-FBG PDH  \u2014 best case (15 MHz, quiet fiber, spliced)",
             FBGParams(**{**p.__dict__, "fwhm_pdh": 15e6,
                          "fiber_thermal_1g": 3.0, "etalon_flicker_pdh": 1.0}), "pdh"),
            ("pi-FBG PDH  \u2014 non-PM fiber",
             FBGParams(**{**p.__dict__, "use_pm_fiber": False}), "pdh"),
            ("std telecom FBG \u2014 side-of-fringe",
             FBGParams(**{**p.__dict__}), "sof"),
        ]
    rows = []
    for name, pp, scheme in cases:
        uc = sigma_allan(pp, tau, scheme, ratio=pp.ratio_cancel)[0]
        U  = k * uc
        rows.append([name,
                     f"{uc:,.2f}", f"{uc/pp.S*1e9:.1f} nK", f"{hz_to_uVV(uc):.2e}",
                     f"{U:,.2f}",  f"{U/pp.S*1e9:.1f} nK",  fmt_V(U), f"{hz_to_uVV(U):.2e}"])
    df = pd.DataFrame(rows, columns=[
        "Scheme (ratiometric)",
        "u_c [Hz]", "u_c [\u0394T]", "u_c [\u00b5V/V]",
        f"U (k={k:g}) [Hz]", "U [\u0394T]", "U [abs]", "U [\u00b5V/V]"])
    num = [c for c in df.columns if c != "Scheme (ratiometric)"]
    return df.style.hide(axis="index").set_properties(subset=num, **{"text-align": "right"})

print(f"FBG VOLTAGE TWIN @ tau_meas = {FBGParams().tau_meas:g} s, quoted at V_FS = {V_FS:g} V")
voltage_summary(FBGParams())

### 8b. Why every voltage number needs an operating point

Unchanged in form from the microring notebook: the slope dν/dV = 2k₁V collapses as the applied voltage falls, so the same frequency uncertainty buys progressively less voltage resolution — **absolute** uncertainty grows as **1/V**, **relative** as **1/V²**. A decade down the range costs 10× in nV and 100× in µV/V. Quote V with the number.

In [ ]:
def fs_penalty(p: FBGParams, scheme="pdh", volts=(2.0, 1.0, 0.5, 0.2, 0.1)):
    uc = sigma_allan(p, p.tau_meas, scheme, ratio=p.ratio_cancel)[0]
    U  = p.k * uc
    rows = [[f"{V:.2f}", f"{K1*V*V/1e9:.3g}", f"{dnu_dV(V)/1e9:.2f}",
             fmt_V(U, V), f"{hz_to_uVV(U, V):.2e}", f"{K_to_uVV(1e-9, p.S, V):.2e}"]
            for V in volts]
    df = pd.DataFrame(rows, columns=["V [V]", "\u03bd(V) [GHz]", "d\u03bd/dV [GHz/V]",
                                     "U [abs]", "U [\u00b5V/V]", "1 nK [\u00b5V/V]"])
    return df.style.hide(axis="index").set_properties(
        subset=list(df.columns[1:]), **{"text-align": "right"})

print("pi-FBG PDH scheme, U at k=2, versus operating point (1/V absolute, 1/V\u00b2 relative)")
fs_penalty(FBGParams())

---
*Voltage result: with a commercial π-FBG and ratiometric cancellation, the expanded uncertainty (k = 2) at τ ≈ 60 s is of order **1.4×10⁻³ µV/V** at the 2 V full scale, improving to roughly **6×10⁻⁴ µV/V** in the best case (15 MHz TeraXion-class grating, all-spliced APC path, quiet fiber) — i.e. the same **~1.4 to 0.6 nV/V** band the microring reached, arrived at from a different direction. The FBG gets there through linewidth rather than through sensitivity, and it gets there with an off-the-shelf part instead of a custom die.*

*The spread is set by fiber thermodynamic noise and parasitic-etalon fringes, not by RAM. Non-PM fiber costs roughly 3× and is not an option. A standard telecom FBG is not an option at all.*

---

## 9. Bill of design consequences

Things this budget says the hardware must do, which the microring version did not require:

1. **Buy π-phase-shifted gratings, matched pair, one draw, one phase mask**, specified to ±10 pm of each other and ≤50 MHz FWHM (15 MHz if the budget allows). Suppliers: TeraXion, AOS GmbH, O/E Land, Technica.
2. **PM fiber throughout**, with an aligned launch and a specified PER. Section 8a prices the alternative.
3. **Add a strain trim** — one PZT stack on one grating, **≥20 µε range** (≈0.2 µm of stretch over a 10 mm grating) to close a ±10 pm matched pair, with margin. Section 2b sizes it. Do **not** try to close the pair mismatch with the TECs: it takes a ~20 K offset between the gratings for a catalogue pair, and that offset *is* the measurand.
4. **All-spliced optical path.** Every connector is a parasitic etalon whose FSR (~100 MHz for a 1 m stub) sits right on top of the π-FBG linewidth. Where a connector is unavoidable, 8° APC. Isolator after each DFB.
5. **Add an EOM** — real PDH needs phase modulation. This is the one component the dither scheme let you skip, and the reason it is worth adding is 38× on the discriminant.
6. **Two separate fibers, one grating each**, on a common substrate — not two gratings in one fiber (§2.4). Keep them **≥ 15 mm apart** (≈ 2× the 60 s thermal diffusion length in silica) with a real thermal barrier: that separation is what makes the differential measurement work *and* what decorrelates the two gratings' thermodynamic noise, which is why that term carries the √2.
7. **Mount for differential strain, and treat η_ε as a spec to be verified.** With two separate fibers the common-strain rejection comes entirely from mount symmetry — symmetric bonding, matched adhesive, matched bond length, common anchors. Measure it; do not assume 1e-3.
8. **Measure, in this order:** fiber thermal-noise floor of the actual pair, parasitic-fringe amplitude of the assembled path, polarisation stability. Section 5b shows how much each one moves the answer.

---
### References for the device numbers used here
- Silica FBG at 1550 nm: **10–12 pm/K** temperature and **≈1.2 pm/µε** strain sensitivity — standard values, widely reported in the FBG-sensing literature.
- **π-phase-shifted FBG at 15 MHz FWHM**, commercially produced (TeraXion) and used as a laser frequency discriminator; π-FBG resonators have been PDH-locked directly, and ~3 MHz resonances demonstrated in tunable π-FBG cavities.
- **Fiber thermodynamic (thermoconductive) noise** as the fundamental floor of fiber-optic frequency references, measured directly in fiber Bragg-grating lasers; thermodynamic noise dominates above ~100 Hz in fiber.

---

## 10. Final table — expanded combined uncertainty, microring vs FBG

The single summary table: **U = k·u_c with k = 2 (≈95 %)**, ratiometric, at **τ = 60 s**, quoted at **V_FS = 2 V**.

The microring model is transcribed inline below (from `laser_driver_uncertainty_budget.ipynb`) so this notebook stands alone and the two columns are computed by the same code path rather than copied by hand.

**Read the kelvin column, not the hertz column.** The two references have different S (2.00 vs 1.25 GHz/K) and different full-scale spans (14.0 vs 8.7 GHz), so hertz is not a common currency between them. Kelvin and µV/V are.

In [ ]:
# --- microring model, transcribed from laser_driver_uncertainty_budget.ipynb ---
S_RING_    = 2.0e9
NU_FS_RING_= 14.0e9
V_FS_      = 2.0

def ring_contributors(ram_flicker=20.0):
    # (name, a_dither@1s, a_rf@1s, mu, ratio_cancels)
    return [
        ("Detection / discrimination", 150.0, 20.0, -0.5, False),
        ("Residual laser FM floor",     50.0,  6.0,  0.0, False),
        ("RIN in-band",                 60.0,  5.0,  0.0, False),
        ("RAM in-band (flicker)", ram_flicker, ram_flicker, 0.0, False),
        ("Carrier-dither smear",        50.0,  0.0, -0.5, False),
        ("Timebase short-term",          1.0,  0.1, -0.5, False),
        ("RAM slow drift",              50.0, 50.0,  0.5, True),
        ("Thermal common-mode leakage", 30.0, 30.0,  0.5, True),
        ("Strain / mount creep",        20.0, 20.0,  0.5, True),
    ]

def ring_sigma(tau, scheme, ram_flicker=20.0, ratio=True):
    tot = 0.0
    for (n, ad, ar, mu, canc) in ring_contributors(ram_flicker):
        if ratio and canc: continue
        a = ar if scheme == "rf" else ad
        tot += (a * tau**mu)**2
    return np.sqrt(tot)

def _volt(hz, nu_fs, V=V_FS_):
    k1 = nu_fs / V_FS_**2
    return hz / (2*k1*V)
def _uVV(hz, nu_fs, V=V_FS_):
    return _volt(hz, nu_fs, V) / V * 1e6

def final_table(tau=60.0, k=2.0):
    rows = []
    def add(fam, case, fwhm, S, nu_fs, uc, note):
        U = k*uc
        rows.append([fam, case, fwhm, f"{S/1e9:.2f}", f"{uc:,.2f}", f"{U:,.2f}",
                     f"{U/S*1e9:,.1f}", f"{_volt(U,nu_fs)*1e9:.2f}",
                     f"{_uVV(U,nu_fs):.2e}", note])

    # microring
    for case, scheme, ram in [
        ("Dither \u2014 in-band RAM survives",   "dither", 20.0),
        ("Dither \u2014 RAM slower than window", "dither",  0.0),
        ("RF \u2014 in-band RAM survives",       "rf",     20.0),
        ("RF \u2014 RAM slower than window",     "rf",      0.0),
    ]:
        add("Microring (SiN, Q=1e5)", case, "1.9 GHz", S_RING_, NU_FS_RING_,
            ring_sigma(tau, scheme, ram), "as-built reference")

    # FBG
    for case, kw, scheme, fwhm, note in [
        ("\u03c0-FBG PDH \u2014 nominal", {}, "pdh", "50 MHz", "baseline design"),
        ("\u03c0-FBG PDH \u2014 best case",
         dict(fwhm_pdh=15e6, fiber_thermal_1g=3.0, etalon_flicker_pdh=1.0),
         "pdh", "15 MHz", "quiet spliced fiber, etalon killed"),
        ("\u03c0-FBG PDH \u2014 worst case",
         dict(fiber_thermal_1g=15.0, etalon_flicker_pdh=15.0, strain_diff_flicker=10.0),
         "pdh", "50 MHz", "all three unknowns bad"),
        ("\u03c0-FBG PDH \u2014 non-PM fiber", dict(use_pm_fiber=False),
         "pdh", "50 MHz", "polarisation-limited"),
        ("Standard telecom FBG \u2014 side-of-fringe", {}, "sof", "25 GHz",
         "catalogue part \u2014 not viable"),
    ]:
        pp = FBGParams(**kw)
        add("Fiber Bragg grating", case, fwhm, pp.S, NU_FS,
            sigma_allan(pp, tau, scheme, ratio=pp.ratio_cancel)[0], note)

    df = pd.DataFrame(rows, columns=[
        "Reference", "Scheme / case", "FWHM", "S [GHz/K]",
        "u_c [Hz]", f"U (k={k:g}) [Hz]", "U [nK]", "U [nV]", "U [\u00b5V/V]", "Note"])
    num = ["S [GHz/K]", "u_c [Hz]", f"U (k={k:g}) [Hz]", "U [nK]", "U [nV]", "U [\u00b5V/V]"]
    return df, df.style.hide(axis="index").set_properties(subset=num, **{"text-align": "right"})

print(f"EXPANDED COMBINED UNCERTAINTY  \u2014  k=2 (~95%), ratiometric, tau = 60 s, at V_FS = 2 V")
print(f"microring nu_FS = {NU_FS_RING_/1e9:.2f} GHz   |   FBG nu_FS = {NU_FS/1e9:.2f} GHz\n")
_df, _sty = final_table()
_sty

### 10a. Markdown export of the final table

In [ ]:
_df, _ = final_table()
print(to_markdown(_df, ["S [GHz/K]", "u_c [Hz]", "U (k=2) [Hz]", "U [nK]",
                        "U [nV]", "U [\u00b5V/V]"]))

---
### Reading the final table

- **The two viable architectures land within ~2× of each other in kelvin.** Microring RF at 21.6 nK (RAM-limited) versus π-FBG PDH at 18.8 nK nominal. Neither is clearly better on paper; they fail differently, which is the real basis for choosing.
- **Best cases are also close** — 8.2 nK for the ring if RAM turns out slower than the window, 9.1 nK for the FBG with a 15 MHz grating in a quiet spliced path. Both are contingent on an unmeasured term, and *that contingency is the honest headline of both budgets*.
- **The FBG's downside is bounded and knowable.** Its worst case (44.8 nK, all three unknowns bad) is a factor of ~2.4 off nominal, and every term in it is attackable with packaging. The microring's dither fallback sits at 83 nK regardless of RAM — the dither scheme's carrier smear and 1.9 GHz linewidth dominate, and no amount of packaging fixes a linewidth.
- **Non-PM fiber (51.5 nK) is worse than the FBG's own worst case.** It is not a cost-saving option; it is a different, worse instrument.
- **The standard telecom FBG at 1.06 µK is off the table** — 56× worse than the π-FBG and 49× worse than the microring RF scheme. It is in the table only to close the question.
- **In µV/V the ranking is the same** because both references share the 2 V full scale, but note the FBG's 1.6× weaker hertz→volt slope partially offsets its better hertz figure: π-FBG PDH nominal is 1.34×10⁻³ µV/V against the ring RF's 1.54×10⁻³, a much narrower gap than the hertz column suggests.

---

## 11. Can the v7 electronics drive an FBG head?

Checked against the as-built v7 cards and the user manual. **Nothing in the crate is a blocker.** The signal chain is scheme-compatible, and the required changes are one resistor per channel, firmware, and optics/mechanics outside the crate.

### 11.1 What works unchanged

| Block | As built | Verdict for FBG |
|---|---|---|
| **Lock scheme** | 2–2.5 MHz laser-current derivative / lock-in discriminator (explicitly *not* textbook PDH) | ✅ Correct regime. Ω/FWHM = 0.05 on a 50 MHz line — still comfortably a slope discriminator. |
| **Modulation injection** | Bias-tee into the laser anode (L 10 µH, C 10 nF, R_inj), deliberately *not* the FET gate | ✅ Ideal. Keeping the FET's nonlinear g_m out of the amplitude path is even more valuable here, since RAM is the FBG's neighbour term. |
| **Photodiode** | **FGA01FC — FC-pigtailed InGaAs**, 12 V bias, C_d 2.13 pF measured | ✅ **Already fiber-coupled.** The optical interface an FBG head wants is the one the AFE already has. |
| **TIA** | OPA814, R_f 8.06 kΩ, C_f 1 pF, f₃dB ≈ 19.7 MHz, −8.06 mV/µA | ✅ Spec is "≥5–10× Ω" and Ω is unchanged. 19.7 MHz ≫ 2.5 MHz still holds. |
| **Demodulator** | AD835 4-quadrant multiplier (250 MHz) × phase-set DDS reference | ✅ Unchanged. The 250 MHz headroom is what would make a *true*-PDH upgrade partly reusable (§11.4). |
| **Post-mix / ERR** | 64–72 kHz pole, ×5.02 to a 2.500 V datum, offset null via AD5696R | ✅ Unchanged. |
| **Servo** | CH-card PI integrator, 0.483 mA/V authority, DAC FS 90.5 mA | ✅ Unchanged. |
| **Beat readout** | **External** PD + external counter; result never re-enters the crate | ✅ Out of scope of the crate either way. |
| **Two channels** | Two independent PD/TIA/demod slices (PD1/2, TIA1/2, ERR1/2) | ✅ Exactly matches the **two-separate-fibers** head that §2.4 now requires. |

### 11.2 The one hardware change: R_inj

Modulation depth is set by `I_pk ≈ V_src,pk / R_inj = 0.6 V / 2 kΩ ≈ 0.30 mA`, chosen to give **0.1–0.3 × the fringe half-width**. That rule is a *fraction of the linewidth*, so a 38× narrower line needs a 38× smaller depth. R_inj is already documented as a trim resistor. The cell below sizes it.

Two things get *better*, not worse, when you retrim:
- the bias-tee inequality `X_C(Ω) ≪ R_inj` is more strongly satisfied as R_inj rises;
- the Johnson current noise injected into the laser goes as 1/√R_inj, so a larger R_inj injects *less* noise.

### 11.3 What has to change outside the crate

1. **Two fibers, not one** (§2.4) — a head with two π-FBGs on separate fibers across a thermal barrier on a common substrate.
2. **Pair trim: make it passive.** §2b sizes ~17 µε for a ±10 pm pair. Do **not** build an active PZT driver channel: an active trim writes its driver noise straight onto the measurand through S_ε = 150 MHz/µε, and there is no HV rail in the crate anyway (±22 V max). Use a **one-time mechanical stretch, locked down** at assembly. This is why the "missing PZT driver" is not a gap.
3. **Do not trim with the TECs.** Both TEC channels are committed to the laser butterflies, and a thermal trim is worse than merely absent: a standing inter-grating temperature offset *is* the measurand, so the TEC servo's own noise would be written directly onto the result.
4. **Connectors — the one place the as-built optics fights the FBG budget.** The manual specifies FC/APC on the laser side and **FC/PC at the detector**. PC endfaces are the parasitic-etalon source that §2.3 identifies as a floor-setting term, and a ~1 m stub's ~100 MHz FSR sits right on top of a 50 MHz grating linewidth. Move the detector side to **APC or fusion splice** if the FBG route is taken. The laser-side APC choice is already correct and matters more than ever.
5. **Reference-head thermal control.** The crate has no TEC channel for the reference — true for the ring die too, but a fiber head is a larger, slower thermal mass with a different gradient structure. Enclosure design, not a card change; there are spare backplane slots if it ever needs a channel.

### 11.4 If you ever want *true* dispersive PDH

Not required — the derivative lock captures the whole linewidth gain — but for completeness, Ω ≳ 50 MHz would need an **EOM** and driver (the crate has none, and laser-current FM cannot reach), a **>50 MHz LO** (the AD9833 on a 25 MHz clock cannot), and a **faster TIA** (19.7 MHz is the binding limit). The AD835 demodulator would survive. That is an AFE respin plus a new modulation source — a different instrument, not a retrofit.

### 11.5 Firmware, not hardware

Acquisition is the real work. The lock feature is **38× narrower**, so:
- the sweep step must fall below ~10 MHz (a few LSB of the 16-bit laser DAC — see the cell below), making a full sweep proportionally slower;
- lock-detect and `thresh` logic keyed to a ~1.9 GHz-wide feature must be retuned to an ~50 MHz one;
- `omega1`/`omega2` stay inside the existing 100 kHz–3.5 MHz range, so no CLI change there.

In [ ]:
# --- v7 hardware compatibility arithmetic ------------------------------------
V_SRC_PK   = 0.6      # AD9833 drive amplitude at the injection resistor [V]
RINJ_BUILT = 2.0e3    # as built [ohm]
I_BUILT    = V_SRC_PK / RINJ_BUILT          # 0.30 mA pk
RING_HWHM  = 1.9e9 / 2

# The as-built depth is specified as 0.1-0.3x the fringe half-width. Invert that
# to bracket the DFB FM coefficient at Omega, then re-solve for the FBG linewidth.
fm_lo = 0.1 * RING_HWHM / I_BUILT     # Hz per amp
fm_hi = 0.3 * RING_HWHM / I_BUILT
print(f"As built: I_pk = {I_BUILT*1e3:.2f} mA through R_inj = {RINJ_BUILT/1e3:.0f}k")
print(f"Implied DFB FM coefficient at Omega: "
      f"{fm_lo*1e-3/1e6:.0f}-{fm_hi*1e-3/1e6:.0f} MHz/mA\n")

def rinj_for(fwhm, label):
    h = fwhm / 2.0
    i_lo = 0.1 * h / fm_hi
    i_hi = 0.3 * h / fm_lo
    return [label, f"{fwhm/1e6:.0f} MHz",
            f"{0.1*h/1e6:.1f}-{0.3*h/1e6:.1f} MHz",
            f"{i_lo*1e6:.1f}-{i_hi*1e6:.1f} uA",
            f"{V_SRC_PK/i_hi/1e3:.0f}-{V_SRC_PK/i_lo/1e3:.0f} k"]

rows = [["microring (as built)", "1900 MHz",
         f"{0.1*RING_HWHM/1e6:.0f}-{0.3*RING_HWHM/1e6:.0f} MHz",
         f"{I_BUILT*1e6:.0f} uA", f"{RINJ_BUILT/1e3:.0f} k"],
        rinj_for(50e6, "pi-FBG, 50 MHz"),
        rinj_for(15e6, "pi-FBG, 15 MHz")]
_d = pd.DataFrame(rows, columns=["Reference", "FWHM", "Target FM depth",
                                 "I_pk needed", "R_inj needed"])
display(_d.style.hide(axis="index").set_properties(
    subset=list(_d.columns[1:]), **{"text-align": "right"}))

# bias-tee sanity at the new R_inj, and injected Johnson noise
OM, L_BT, C_BT = 2.0e6, 10e-6, 10e-9
XC = 1/(2*np.pi*OM*C_BT); XL = 2*np.pi*OM*L_BT
kB, T = 1.380649e-23, 300.0
print(f"\nBias-tee at Omega=2 MHz: X_L = {XL:.0f} ohm (>> Rsense+r_laser), "
      f"X_C = {XC:.0f} ohm (<< R_inj)")
print("  both inequalities hold HARDER as R_inj rises:")
for R in (RINJ_BUILT, 75e3):
    print(f"    R_inj = {R/1e3:5.0f}k -> Johnson noise "
          f"{np.sqrt(4*kB*T/R)*1e12:.2f} pA/rtHz injected into the laser")

# --- acquisition: how many DAC codes wide is the lock feature? ---------------
DAC_BITS, I_FS = 16, 90.5e-3      # AD5696R, CH-card laser DAC full scale
lsb_A = I_FS / 2**DAC_BITS
print(f"\nLaser DAC: {DAC_BITS}-bit over {I_FS*1e3:.1f} mA -> {lsb_A*1e6:.2f} uA/LSB")
for tune in (1.0, 2.0, 3.0):
    hz_lsb = lsb_A*1e3 * tune * 1e9
    print(f"  DC tuning {tune:.0f} GHz/mA -> {hz_lsb/1e6:5.2f} MHz/LSB  |  "
          f"1.9 GHz line = {1.9e9/hz_lsb:6.0f} LSB  |  "
          f"50 MHz line = {50e6/hz_lsb:5.0f} LSB")
print("\nThe 50 MHz feature is still tens of codes wide -- catchable, but the sweep")
print("step and lock-detect thresholds must be retuned. Firmware, not hardware.")

**Bottom line for §11:** the v7 crate can lock an FBG head. The changes are `R_inj` (one resistor per channel, ~2 k → ~75 k), acquisition firmware, and the optical head itself — plus moving the detector-side connector off FC/PC, which is the one as-built choice that actively fights the FBG error budget.